# Debug Session

This notebook demonstrates the `DebugSession` API for interactive build debugging.
It sets up the full build environment without running the build, then lets you
iteratively run and re-run the build script, inspecting output each time.

This is ideal for:

- Debugging build failures interactively
- AI agents that need to fix and retry builds programmatically
- Understanding what happens during a build step by step

In [ ]:
from rattler_build import (
    DebugSession,
    RenderConfig,
    Stage0Recipe,
    VariantConfig,
)

## Parse and Render a Recipe

Create a simple recipe with an inline build script and render it:

In [ ]:
recipe_yaml = """
package:
  name: debug-demo
  version: "1.0.0"

build:
  number: 0
  noarch: generic
  script:
    content: |
      echo "Hello from the build script!"
      mkdir -p $PREFIX/bin
      echo "hello from debug-demo" > $PREFIX/bin/hello

requirements:
  run:
    - python
"""

recipe = Stage0Recipe.from_yaml(recipe_yaml)
variants = recipe.render(VariantConfig(), RenderConfig())
print(f"Rendered {len(variants)} variant(s)")

## Create the Debug Session

`DebugSession.create()` resolves dependencies, fetches sources, installs
environments, and creates the build script — all without running the actual build.

In [ ]:
session = DebugSession.create(
    variants[0],
    channels=["conda-forge"],
)

print(f"Work directory:  {session.work_dir}")

## Read the Build Script

Before running, inspect the generated build script:

In [ ]:
print("Build script contents:")
print(session.read_build_script())

print("\nWork directory contents:")
for p in sorted(session.work_dir.iterdir()):
    print(f"  {p.name}")

## Run the Build Script

Run with `trace=True` to see each command as it executes:

In [ ]:
result = session.run_script(trace=True)

print(f"Exit code: {result.exit_code}")
print(f"Success:   {result.success}")
print(f"\nStdout:\n{result.stdout}")
if result.stderr:
    print(f"\nStderr:\n{result.stderr}")

## Iterate — Modify and Re-run

The key advantage: modify the build script and re-run without repeating
the expensive dependency resolution and environment setup.

In [ ]:
# Modify the build script
content = session.build_script.read_text()
content = content.replace(
    'echo "Hello from the build script!"',
    'echo "Hello from the MODIFIED build script!"',
)
session.build_script.write_text(content)

# Re-run — fast since environments are already installed
result = session.run_script(trace=True)
print(f"Exit code: {result.exit_code}")
print(f"Stdout:\n{result.stdout}")

## Error Handling for AI Agents

For automated workflows, `ScriptResult` gives structured access to the output.
Non-zero exit codes are values, not exceptions — making it easy to write
retry loops.

In [ ]:
result = session.run_script()

if not result.success:
    print("Build failed!")
    print(f"stderr: {result.stderr}")
    if "command not found" in result.stderr:
        print("Missing command — may need a build dependency")
    elif "No such file or directory" in result.stderr:
        print("Missing file — check source extraction")
else:
    print("Build succeeded!")

## Summary

Use `DebugSession.create(variant)` to set up a full build environment, then call `run_script()` to execute the build. Modify files and re-run without repeating the expensive setup. See also `add_packages()` and `create_patch()` in the [API reference](../reference/debug.md).
